# Huấn luyện Hệ thống Nhận diện Phương tiện & Biển số Edge Dashcam (D-FINE + SVTRv2)

Pipeline 3 module tối ưu cho phần cứng biên:
* **Module 1 (`Vehicle Detector` — 1 lớp `vehicle`, `num_queries=35`):** `D-FINE` (ICLR 2025, NMS-Free) phát hiện toàn bộ phương tiện trên khung hình `(384, 640)`.
* **Module 2 (`Plate ROI Detector` — 1 lớp `license_plate`, `num_queries=15`):** `D-FINE` chạy trên vùng cắt thân dưới của Top 1–2 xe gần nhất `(256, 384)` — nhanh gấp ~10 lần so với chạy toàn khung hình.
* **Module 3 (`Sequence Plate OCR` — Đọc biển 1 dòng & 2 dòng đầu cuối):** `SVTRv2` (ICCV 2025, `CTCLoss`) xuất trực tiếp `log_probs` dạng `(T, B, C)`.

In [1]:
import os
import re
import sys
import time
import math
import copy
import shutil
import inspect
import random
import yaml
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageFilter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from torchvision.ops import box_iou

_cwd = Path.cwd().resolve()
_search_dirs = [_cwd, *_cwd.parents]

PROJECT_ROOT = next(
    (p for p in _search_dirs if (p / "datasets").exists() or (p / "src").exists()),
    _cwd,
)
MODELS_PARENT = next(
    (p for p in _search_dirs if (p / "models_sota").exists()),
    PROJECT_ROOT / "src" if (PROJECT_ROOT / "src/models_sota").exists() else PROJECT_ROOT,
)

for _p in [str(PROJECT_ROOT), str(MODELS_PARENT)]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from models_sota.dfine_detector import build_dfine_detector, DFINECriterion
from models_sota.svtrv2_ocr import (
    build_svtrv2_ocr,
    unwrap_two_line_plate,
    ctc_greedy_decode,
    PlateVoteAggregator,
)
from models_sota.boost_tracker import (
    DashcamHybridTracker,
    select_priority_tracks,
    extract_plate_search_roi,
    compute_plate_sharpness,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True

print(f"Thư mục gốc dự án: {PROJECT_ROOT}")
print(f"Thiết bị huấn luyện: {DEVICE} ({torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'})")

NUM_WORKERS = 0 if os.name == "nt" else 2
PIN_MEMORY = (DEVICE.type == "cuda")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# True: gộp 4 lớp xe về 1 lớp 'vehicle' khớp thiết kế DashcamHybridTracker trong boost_tracker.py
# False: giữ nguyên 4 lớp ['car', 'motorcycle', 'bus', 'truck']
SINGLE_CLASS_VEHICLE = True
VEHICLE_CLASSES = ["vehicle"] if SINGLE_CLASS_VEHICLE else ["car", "motorcycle", "bus", "truck"]
PLATE_CLASSES = ["license_plate"]

DETECTOR_VARIANT = "dfine_s"
OCR_VARIANT = "svtrv2_base"

PROCESSED_DIR = (PROJECT_ROOT / "datasets/processed").resolve()
SAVE_DIR = (PROJECT_ROOT / "runs/sota_alpr").resolve()
SAVE_DIR.mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(42)

Thư mục gốc dự án: E:\DuAn\Tiny-model-for-number-plate-recognization
Thiết bị huấn luyện: cuda (NVIDIA GeForce RTX 4060 Laptop GPU)


In [2]:
def _link_or_copy(src: Path, dst: Path):
    if dst.exists():
        return
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def _read_yolo_boxes(lbl_path: Path):
    items = []
    if not lbl_path.exists():
        return items
    with open(lbl_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                cid = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:5])
                items.append((cid, xc, yc, bw, bh))
    return items


def build_single_class_vehicle_dataset(src_root: Path, dst_root: Path) -> Path:
    """Gộp 4 lớp xe (0..3) thành 1 lớp duy nhất 'vehicle' (0)."""
    for split in ["train", "val"]:
        src_img_dir = src_root / split / "images"
        src_lbl_dir = src_root / split / "labels"
        if not src_img_dir.exists():
            continue

        dst_img_dir = dst_root / split / "images"
        dst_lbl_dir = dst_root / split / "labels"
        dst_img_dir.mkdir(parents=True, exist_ok=True)
        dst_lbl_dir.mkdir(parents=True, exist_ok=True)

        for img_path in sorted(src_img_dir.iterdir()):
            if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
                continue
            _link_or_copy(img_path, dst_img_dir / img_path.name)
            boxes = _read_yolo_boxes(src_lbl_dir / f"{img_path.stem}.txt")
            lines = [f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}" for _, xc, yc, bw, bh in boxes]
            (dst_lbl_dir / f"{img_path.stem}.txt").write_text("\n".join(lines), encoding="utf-8")

    print(f"[Module 1 - 1 Class] Hoàn tất tại: {dst_root}")
    return dst_root


def build_plate_roi_dataset(
    plate_det_root: Path,
    vehicle_det_root: Path,
    dst_roi_root: Path,
    min_plate_vis: float = 0.65,
) -> Path:
    """
    Cắt ROI thân dưới xe từ ảnh gốc theo đúng hàm extract_plate_search_roi() để train Module 2 siêu nhanh.
    Nếu ảnh biển số chưa có file nhãn xe đi kèm, tự động tạo vùng crop thân dưới giả lập quanh biển số.
    """
    rng = random.Random(42)

    for split in ["train", "val"]:
        is_train = (split == "train")
        p_img_dir = plate_det_root / split / "images"
        p_lbl_dir = plate_det_root / split / "labels"
        v_lbl_dir = vehicle_det_root / split / "labels"
        if not p_img_dir.exists():
            continue

        out_img_dir = dst_roi_root / split / "images"
        out_lbl_dir = dst_roi_root / split / "labels"
        out_img_dir.mkdir(parents=True, exist_ok=True)
        out_lbl_dir.mkdir(parents=True, exist_ok=True)

        saved_crops = 0
        saved_plates = 0

        for img_path in sorted(p_img_dir.iterdir()):
            if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
                continue

            plate_items = _read_yolo_boxes(p_lbl_dir / f"{img_path.stem}.txt")
            if len(plate_items) == 0:
                continue

            img_bgr = cv2.imread(str(img_path))
            if img_bgr is None:
                continue
            H, W = img_bgr.shape[:2]

            # Quy đổi plate boxes sang pixel xyxy
            plate_xyxy = []
            for _, xc, yc, bw, bh in plate_items:
                px1 = (xc - 0.5 * bw) * W
                py1 = (yc - 0.5 * bh) * H
                px2 = (xc + 0.5 * bw) * W
                py2 = (yc + 0.5 * bh) * H
                if (px2 - px1) >= 6 and (py2 - py1) >= 4:
                    plate_xyxy.append([px1, py1, px2, py2])
            if not plate_xyxy:
                continue
            plate_xyxy = np.array(plate_xyxy, dtype=np.float32)

            # Đọc vehicle boxes nếu tồn tại cùng tên file
            veh_items = _read_yolo_boxes(v_lbl_dir / f"{img_path.stem}.txt")
            roi_boxes = []

            if len(veh_items) > 0:
                for _, vxc, vyc, vbw, vbh in veh_items:
                    vx1 = (vxc - 0.5 * vbw) * W
                    vy1 = (vyc - 0.5 * vbh) * H
                    vx2 = (vxc + 0.5 * vbw) * W
                    vy2 = (vyc + 0.5 * vbh) * H

                    # Thêm jitter nhẹ lúc train để mô phỏng sai số khung dự đoán từ tracker
                    trim = rng.uniform(0.20, 0.30) if is_train else 0.25
                    pad = rng.uniform(0.05, 0.12) if is_train else 0.08
                    _, (rx1, ry1, rx2, ry2) = extract_plate_search_roi(
                        img_bgr, np.array([vx1, vy1, vx2, vy2]), top_trim_ratio=trim, pad_ratio=pad
                    )
                    if (rx2 - rx1) >= 32 and (ry2 - ry1) >= 24:
                        roi_boxes.append((rx1, ry1, rx2, ry2))
            else:
                # Fallback: tạo vùng crop thân dưới phương tiện bao quanh biển số
                for px1, py1, px2, py2 in plate_xyxy:
                    pw, ph = px2 - px1, py2 - py1
                    scale_w = rng.uniform(3.2, 5.5) if is_train else 4.2
                    scale_h = rng.uniform(3.2, 5.5) if is_train else 4.2
                    roi_w, roi_h = pw * scale_w, ph * scale_h
                    cx = 0.5 * (px1 + px2) + (rng.uniform(-0.18, 0.18) * roi_w if is_train else 0.0)
                    cy = 0.5 * (py1 + py2) - (rng.uniform(0.05, 0.20) * roi_h if is_train else 0.12 * roi_h)
                    rx1 = max(0, int(round(cx - 0.5 * roi_w)))
                    ry1 = max(0, int(round(cy - 0.5 * roi_h)))
                    rx2 = min(W, int(round(cx + 0.5 * roi_w)))
                    ry2 = min(H, int(round(cy + 0.5 * roi_h)))
                    if (rx2 - rx1) >= 32 and (ry2 - ry1) >= 24:
                        roi_boxes.append((rx1, ry1, rx2, ry2))

            neg_kept = 0
            for r_idx, (rx1, ry1, rx2, ry2) in enumerate(roi_boxes):
                rw, rh = float(rx2 - rx1), float(ry2 - ry1)
                yolo_lines = []

                for px1, py1, px2, py2 in plate_xyxy:
                    orig_area = max((px2 - px1) * (py2 - py1), 1.0)
                    ix1 = max(float(rx1), px1)
                    iy1 = max(float(ry1), py1)
                    ix2 = min(float(rx2), px2)
                    iy2 = min(float(ry2), py2)
                    if ix2 <= ix1 or iy2 <= iy1:
                        continue
                    vis = ((ix2 - ix1) * (iy2 - iy1)) / orig_area
                    if vis >= min_plate_vis:
                        nxc = ((ix1 + ix2) * 0.5 - rx1) / rw
                        nyc = ((iy1 + iy2) * 0.5 - ry1) / rh
                        nbw = (ix2 - ix1) / rw
                        nbh = (iy2 - iy1) / rh
                        if nbw > 0.01 and nbh > 0.01:
                            yolo_lines.append(f"0 {nxc:.6f} {nyc:.6f} {nbw:.6f} {nbh:.6f}")

                # Giữ tối đa 1 ROI âm tính mỗi ảnh trên tập train để chống nhận diện nhầm lưới tản nhiệt/đèn xe
                if len(yolo_lines) == 0:
                    if not (is_train and neg_kept < 1 and rng.random() < 0.20):
                        continue
                    neg_kept += 1

                crop_bgr = img_bgr[ry1:ry2, rx1:rx2]
                out_stem = f"{img_path.stem}_roi{r_idx}"
                cv2.imwrite(str(out_img_dir / f"{out_stem}.jpg"), crop_bgr)
                (out_lbl_dir / f"{out_stem}.txt").write_text("\n".join(yolo_lines), encoding="utf-8")
                saved_crops += 1
                saved_plates += len(yolo_lines)

        print(f"[Module 2 ROI - {split}] Đã tạo {saved_crops} ảnh ROI ({saved_plates} nhãn biển số)")

    return dst_roi_root


M1_DATA_ROOT = PROCESSED_DIR / "module1_vehicle_det"
if SINGLE_CLASS_VEHICLE and M1_DATA_ROOT.exists():
    M1_DATA_ROOT = build_single_class_vehicle_dataset(
        M1_DATA_ROOT, PROCESSED_DIR / "module1_vehicle_1cls_det"
    )

M2_RAW_ROOT = PROCESSED_DIR / "module2_plate_det"
M2_ROI_ROOT = PROCESSED_DIR / "module2_plate_roi_det"
if M2_RAW_ROOT.exists():
    M2_ROI_ROOT = build_plate_roi_dataset(
        plate_det_root=M2_RAW_ROOT,
        vehicle_det_root=PROCESSED_DIR / "module1_vehicle_det",
        dst_roi_root=M2_ROI_ROOT,
    )

[Module 1 - 1 Class] Hoàn tất tại: E:\DuAn\Tiny-model-for-number-plate-recognization\datasets\processed\module1_vehicle_1cls_det
[Module 2 ROI - train] Đã tạo 2059 ảnh ROI (2104 nhãn biển số)
[Module 2 ROI - val] Đã tạo 609 ảnh ROI (614 nhãn biển số)


## Phần 1: Dataset, Augmentation, Model EMA & Pipeline Huấn luyện D-FINE (ICLR 2025 — NMS-Free) cho Module 1 & Module 2

In [3]:
class ModelEMA:
    def __init__(self, model: nn.Module, decay: float = 0.9995, tau: float = 2000.0):
        self.ema_model = copy.deepcopy(model).eval()
        for p in self.ema_model.parameters():
            p.requires_grad_(False)
        self.decay = decay
        self.tau = tau
        self.updates = 0

    @torch.no_grad()
    def update(self, model: nn.Module):
        self.updates += 1
        d = self.decay * (1.0 - math.exp(-self.updates / self.tau))
        msd = model.state_dict()
        for k, v in self.ema_model.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(msd[k].detach(), alpha=1.0 - d)
            else:
                v.copy_(msd[k])


class DFINEDetectionDataset(Dataset):
    def __init__(
        self,
        img_dir: Path,
        lbl_dir: Path,
        num_classes: int,
        img_size=(384, 640),
        is_train=True,
        cache_ram=True,
    ):
        self.img_dir = Path(img_dir)
        self.lbl_dir = Path(lbl_dir)
        self.num_classes = num_classes
        self.img_size = tuple(img_size)
        self.is_train = is_train
        self.cache_ram = cache_ram
        self.cached_imgs = []
        self.cached_labels = []

        if not self.img_dir.exists():
            raise FileNotFoundError(f"Không tìm thấy thư mục ảnh: {self.img_dir}")

        self.img_paths = sorted([
            p for p in self.img_dir.iterdir()
            if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
        ])

        target_h, target_w = self.img_size
        total_bytes = 0
        for img_path in self.img_paths:
            lbl_path = self.lbl_dir / f"{img_path.stem}.txt"
            boxes, labels = [], []
            if lbl_path.exists():
                with open(lbl_path, "r", encoding="utf-8") as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) >= 5:
                            cls_id = int(parts[0])
                            if not (0 <= cls_id < self.num_classes):
                                raise ValueError(
                                    f"class_id={cls_id} trong {lbl_path.name} nằm ngoài [0, {self.num_classes - 1}]"
                                )
                            xc, yc, bw, bh = map(float, parts[1:5])
                            x1 = max(0.0, min(1.0, xc - 0.5 * bw))
                            y1 = max(0.0, min(1.0, yc - 0.5 * bh))
                            x2 = max(0.0, min(1.0, xc + 0.5 * bw))
                            y2 = max(0.0, min(1.0, yc + 0.5 * bh))
                            bw_c, bh_c = x2 - x1, y2 - y1
                            if bw_c > 0.002 and bh_c > 0.002:
                                boxes.append([(x1 + x2) * 0.5, (y1 + y2) * 0.5, bw_c, bh_c])
                                labels.append(cls_id)
            self.cached_labels.append((boxes, labels))

            if self.cache_ram:
                with Image.open(img_path) as im:
                    img = im.convert("RGB")
                img = TF.resize(img, [target_h, target_w])
                self.cached_imgs.append(img)
                total_bytes += target_h * target_w * 3

        if self.cache_ram:
            ram_mb = total_bytes / (1024 ** 2)
            split_tag = "Train" if is_train else "Val"
            print(f"   [{split_tag}] Đã cache {len(self.img_paths)} ảnh ({target_h}x{target_w}) vào RAM (~{ram_mb:.1f} MB)")

    def __len__(self):
        return len(self.img_paths)

    def _random_zoom_out(self, img: Image.Image, boxes: list):
        scale = random.uniform(0.55, 0.95)
        h, w = self.img_size
        new_h, new_w = max(1, int(h * scale)), max(1, int(w * scale))
        img_small = TF.resize(img, [new_h, new_w])
        pad_top = random.randint(0, h - new_h)
        pad_left = random.randint(0, w - new_w)
        canvas = Image.new("RGB", (w, h), (114, 114, 114))
        canvas.paste(img_small, (pad_left, pad_top))
        new_boxes = [
            [
                (b[0] * new_w + pad_left) / w,
                (b[1] * new_h + pad_top) / h,
                b[2] * scale,
                b[3] * scale,
            ]
            for b in boxes
        ]
        return canvas, new_boxes

    def _random_zoom_in_crop(self, img: Image.Image, boxes: list, labels: list):
        h, w = self.img_size
        crop_scale = random.uniform(0.75, 0.95)
        ch, cw = int(h * crop_scale), int(w * crop_scale)
        top = random.randint(0, h - ch)
        left = random.randint(0, w - cw)

        x_min_n, y_min_n = left / w, top / h
        x_max_n, y_max_n = (left + cw) / w, (top + ch) / h

        new_boxes, new_labels = [], []
        for b, lbl in zip(boxes, labels):
            xc, yc, bw, bh = b
            if x_min_n <= xc <= x_max_n and y_min_n <= yc <= y_max_n:
                bx1 = max(x_min_n, xc - 0.5 * bw)
                by1 = max(y_min_n, yc - 0.5 * bh)
                bx2 = min(x_max_n, xc + 0.5 * bw)
                by2 = min(y_max_n, yc + 0.5 * bh)
                nbw = (bx2 - bx1) / crop_scale
                nbh = (by2 - by1) / crop_scale
                if nbw > 0.005 and nbh > 0.005:
                    nxc = ((bx1 + bx2) * 0.5 - x_min_n) / crop_scale
                    nyc = ((by1 + by2) * 0.5 - y_min_n) / crop_scale
                    new_boxes.append([nxc, nyc, nbw, nbh])
                    new_labels.append(lbl)

        if len(new_boxes) == 0:
            return img, boxes, labels

        img_cropped = TF.crop(img, top, left, ch, cw)
        img_cropped = TF.resize(img_cropped, [h, w])
        return img_cropped, new_boxes, new_labels

    def __getitem__(self, idx):
        if self.cache_ram:
            img = self.cached_imgs[idx].copy()
        else:
            with Image.open(self.img_paths[idx]) as im:
                img = im.convert("RGB")
            img = TF.resize(img, list(self.img_size))

        raw_boxes, raw_labels = self.cached_labels[idx]
        boxes = [b[:] for b in raw_boxes]
        labels = raw_labels[:]

        if self.is_train:
            if random.random() < 0.5 and len(boxes) > 0:
                img = TF.hflip(img)
                boxes = [[1.0 - b[0], b[1], b[2], b[3]] for b in boxes]

            r_scale = random.random()
            if r_scale < 0.50 and len(boxes) > 0:
                img, boxes = self._random_zoom_out(img, boxes)
            elif r_scale < 0.85 and len(boxes) > 0:
                img, boxes, labels = self._random_zoom_in_crop(img, boxes, labels)

            if random.random() < 0.65:
                img = TF.adjust_brightness(img, random.uniform(0.65, 1.35))
                img = TF.adjust_contrast(img, random.uniform(0.70, 1.30))
                img = TF.adjust_saturation(img, random.uniform(0.70, 1.30))
                img = TF.adjust_hue(img, random.uniform(-0.04, 0.04))

            if random.random() < 0.10:
                img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.3, 1.0)))

        img_tensor = TF.to_tensor(img)
        img_tensor = TF.normalize(img_tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD)

        if len(boxes) == 0:
            boxes_tensor = torch.zeros((0, 4), dtype=torch.float32)
            labels_tensor = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes_tensor = torch.tensor(boxes, dtype=torch.float32)
            labels_tensor = torch.tensor(labels, dtype=torch.int64)

        return img_tensor, {"boxes": boxes_tensor, "labels": labels_tensor}


def dfine_collate_fn(batch):
    imgs, targets = zip(*batch)
    return torch.stack(imgs, dim=0), list(targets)


def _compute_single_class_ap(tp_list, scores_list, total_gt: int, eval_conf_thresh: float = 0.45):
    if total_gt == 0 or len(scores_list) == 0:
        return 0.0, 0.0, 0.0

    scores_np = np.array(scores_list, dtype=np.float64)
    order = np.argsort(-scores_np)
    scores_sorted = scores_np[order]
    tp = np.array(tp_list, dtype=np.float64)[order]
    fp = 1.0 - tp

    tp_cum = np.cumsum(tp)
    fp_cum = np.cumsum(fp)

    recalls = tp_cum / total_gt
    precisions = tp_cum / np.maximum(tp_cum + fp_cum, 1e-8)

    mrec = np.concatenate(([0.0], recalls, [1.0]))
    mpre = np.concatenate(([0.0], precisions, [0.0]))
    for i in range(len(mpre) - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    ap = float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))

    valid_mask = scores_sorted >= eval_conf_thresh
    if np.any(valid_mask):
        last_idx = np.where(valid_mask)[0][-1]
        eval_prec = float(precisions[last_idx])
        eval_rec = float(recalls[last_idx])
    else:
        eval_prec, eval_rec = 0.0, 0.0

    return ap, eval_prec, eval_rec


@torch.no_grad()
def evaluate_dfine_map50(
    model,
    val_loader,
    device,
    num_classes: int,
    iou_thresh: float = 0.5,
    score_thresh: float = 0.05,
    eval_conf_thresh: float = 0.45,
):
    model.eval()
    cls_tp = {c: [] for c in range(num_classes)}
    cls_scores = {c: [] for c in range(num_classes)}
    cls_gt_count = {c: 0 for c in range(num_classes)}

    for imgs, targets in val_loader:
        _, _, h, w = imgs.shape
        outputs = model.predict(imgs.to(device), score_thresh=score_thresh)

        for out, tgt in zip(outputs, targets):
            gt_cxcywh = tgt["boxes"].to(device)
            gt_labels = tgt["labels"].to(device)

            for lbl in gt_labels.tolist():
                if 0 <= lbl < num_classes:
                    cls_gt_count[lbl] += 1

            if len(gt_cxcywh) > 0:
                gt_boxes = torch.stack([
                    (gt_cxcywh[:, 0] - 0.5 * gt_cxcywh[:, 2]) * w,
                    (gt_cxcywh[:, 1] - 0.5 * gt_cxcywh[:, 3]) * h,
                    (gt_cxcywh[:, 0] + 0.5 * gt_cxcywh[:, 2]) * w,
                    (gt_cxcywh[:, 1] + 0.5 * gt_cxcywh[:, 3]) * h,
                ], dim=-1)
            else:
                gt_boxes = torch.zeros((0, 4), device=device)

            pred_boxes = out["boxes"]
            pred_scores = out["scores"]
            pred_labels = out["labels"]

            if len(pred_boxes) == 0:
                continue

            if pred_labels.max().item() >= num_classes:
                pred_labels = pred_labels - 1

            order = torch.argsort(pred_scores, descending=True)
            pred_boxes = pred_boxes[order]
            pred_scores = pred_scores[order]
            pred_labels = pred_labels[order]

            matched_gt = set()
            if len(gt_boxes) > 0:
                ious = box_iou(pred_boxes, gt_boxes)
                for i in range(len(pred_boxes)):
                    c = int(pred_labels[i].item())
                    if not (0 <= c < num_classes):
                        continue
                    score_val = float(pred_scores[i].item())
                    cls_scores[c].append(score_val)

                    same_cls_mask = (gt_labels == c)
                    if torch.any(same_cls_mask):
                        cls_ious = ious[i].clone()
                        cls_ious[~same_cls_mask] = -1.0
                        best_iou, best_j = cls_ious.max(0)
                        best_j = int(best_j.item())
                        if best_iou.item() >= iou_thresh and best_j not in matched_gt:
                            cls_tp[c].append(1)
                            matched_gt.add(best_j)
                        else:
                            cls_tp[c].append(0)
                    else:
                        cls_tp[c].append(0)
            else:
                for i in range(len(pred_scores)):
                    c = int(pred_labels[i].item())
                    if 0 <= c < num_classes:
                        cls_scores[c].append(float(pred_scores[i].item()))
                        cls_tp[c].append(0)

    ap_per_class = {}
    prec_list, rec_list = [], []
    for c in range(num_classes):
        if cls_gt_count[c] > 0:
            ap_c, prec_c, rec_c = _compute_single_class_ap(
                cls_tp[c], cls_scores[c], cls_gt_count[c], eval_conf_thresh=eval_conf_thresh
            )
            ap_per_class[c] = ap_c
            prec_list.append(prec_c)
            rec_list.append(rec_c)
        else:
            ap_per_class[c] = 0.0

    valid_aps = [ap_per_class[c] for c in range(num_classes) if cls_gt_count[c] > 0]
    if len(valid_aps) == 0:
        return 0.0, 0.0, 0.0, ap_per_class

    return float(np.mean(valid_aps)), float(np.mean(prec_list)), float(np.mean(rec_list)), ap_per_class


def train_dfine_module(
    name: str,
    data_root: Path,
    num_classes: int,
    class_names=None,
    epochs: int = 80,
    warmup_epochs: int = 5,
    batch_size: int = 16,
    lr: float = 2e-4,
    img_size=(384, 640),
    num_queries: int | None = None,
    ema_decay: float = 0.9995,
    resume: bool = False,
):
    if class_names is None:
        class_names = [f"class_{i}" for i in range(num_classes)]

    train_ds = DFINEDetectionDataset(
        data_root / "train/images",
        data_root / "train/labels",
        num_classes=num_classes,
        img_size=img_size,
        is_train=True,
        cache_ram=True,
    )
    val_ds = DFINEDetectionDataset(
        data_root / "val/images",
        data_root / "val/labels",
        num_classes=num_classes,
        img_size=img_size,
        is_train=False,
        cache_ram=True,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=dfine_collate_fn,
        drop_last=(len(train_ds) > batch_size),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=dfine_collate_fn,
    )

    model = build_dfine_detector(
        num_classes=num_classes,
        variant=DETECTOR_VARIANT,
        pretrained_backbone=True,
        num_queries=num_queries,
    ).to(DEVICE)
    criterion = DFINECriterion(num_classes=num_classes).to(DEVICE)
    ema = ModelEMA(model, decay=ema_decay)

    backbone_params = [p for n, p in model.named_parameters() if "backbone" in n and p.requires_grad]
    head_params = [p for n, p in model.named_parameters() if "backbone" not in n and p.requires_grad]
    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_params, "lr": lr * 0.2},
            {"params": head_params, "lr": lr},
        ],
        weight_decay=1e-4,
    )

    warmup_epochs = min(warmup_epochs, max(1, epochs // 4))
    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.05, end_factor=1.0, total_iters=warmup_epochs
    )
    cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(1, epochs - warmup_epochs), eta_min=lr * 0.01
    )
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs]
    )

    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

    out_dir = SAVE_DIR / name
    out_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt = out_dir / f"{name}_{DETECTOR_VARIANT}_best.pth"
    last_ckpt = out_dir / f"{name}_{DETECTOR_VARIANT}_last.pth"

    start_epoch = 1
    best_ap = -1.0
    if resume and last_ckpt.exists():
        ckpt = torch.load(last_ckpt, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["raw_model_state"])
        ema.ema_model.load_state_dict(ckpt["model_state"])
        if "optimizer_state" in ckpt:
            optimizer.load_state_dict(ckpt["optimizer_state"])
        if "scheduler_state" in ckpt:
            scheduler.load_state_dict(ckpt["scheduler_state"])
        start_epoch = int(ckpt.get("epoch", 0)) + 1
        best_ap = float(ckpt.get("best_map50", -1.0))
        print(f"Tiếp tục huấn luyện [{name}] từ Epoch {start_epoch} (Best mAP@50: {best_ap:.4f})")

    history = {
        "epoch": [],
        "lr": [],
        "train_loss": [],
        "val_map50": [],
        "precision": [],
        "recall": [],
    }
    for cname in class_names:
        history[f"ap50_{cname}"] = []

    total_params = sum(p.numel() for p in model.parameters())
    print(
        f"Huấn luyện [{name}] | SOTA: {DETECTOR_VARIANT.upper()} | Queries: {model.num_queries} | "
        f"ImgSize: {img_size} | BS: {batch_size} | Epochs: {start_epoch}..{epochs} | "
        f"Params: {total_params:,} | Train: {len(train_ds)} | Val: {len(val_ds)}"
    )

    for epoch in range(start_epoch, epochs + 1):
        model.train()
        total_loss = 0.0
        t0 = time.perf_counter()

        for step, (imgs, targets) in enumerate(train_loader, 1):
            imgs = imgs.to(DEVICE, non_blocking=True)
            targets = [{k: v.to(DEVICE, non_blocking=True) for k, v in t.items()} for t in targets]

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                outputs = model(imgs, targets)
                loss = criterion(outputs, targets)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            ema.update(model)
            total_loss += loss.item()
            if step % 50 == 0 or step == len(train_loader):
                print(f"\r   Epoch {epoch:02d}/{epochs:02d} [{step}/{len(train_loader)}] - Loss: {loss.item():.4f}", end="")

        scheduler.step()
        current_lr = optimizer.param_groups[1]["lr"]
        avg_loss = total_loss / max(1, len(train_loader))

        ap50, prec, rec, ap_per_class = evaluate_dfine_map50(
            ema.ema_model, val_loader, DEVICE, num_classes=num_classes
        )
        dt = time.perf_counter() - t0

        history["epoch"].append(epoch)
        history["lr"].append(current_lr)
        history["train_loss"].append(avg_loss)
        history["val_map50"].append(ap50)
        history["precision"].append(prec)
        history["recall"].append(rec)
        for c_idx, cname in enumerate(class_names):
            history[f"ap50_{cname}"].append(ap_per_class.get(c_idx, 0.0))

        cls_detail = ""
        if num_classes > 1:
            cls_detail = " | " + " ".join(
                [f"{class_names[c]}:{ap_per_class[c]:.3f}" for c in range(num_classes)]
            )

        print(
            f"\rEpoch [{epoch:02d}/{epochs:02d}] ({dt:.1f}s) | LR: {current_lr:.2e} | "
            f"Loss: {avg_loss:.4f} | Val mAP@50: {ap50:.4f} | P@0.45: {prec:.4f} | R@0.45: {rec:.4f}{cls_detail}"
        )

        ckpt_payload = {
            "variant": DETECTOR_VARIANT,
            "num_classes": num_classes,
            "num_queries": model.num_queries,
            "class_names": class_names,
            "label_index_base": 0,
            "img_size": img_size,
            "mean": IMAGENET_MEAN,
            "std": IMAGENET_STD,
            "epoch": epoch,
            "best_map50": max(best_ap, ap50),
            "model_state": ema.ema_model.state_dict(),
            "raw_model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
        }
        torch.save(ckpt_payload, last_ckpt)

        if ap50 > best_ap:
            best_ap = ap50
            torch.save(ckpt_payload, best_ckpt)

    pd.DataFrame(history).to_csv(out_dir / "history.csv", index=False)

    fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
    ax[0].plot(history["epoch"], history["train_loss"], label="D-FINE Loss", color="#1f77b4")
    ax[0].set_title(f"{name} ({DETECTOR_VARIANT}) - Training Loss")
    ax[0].set_xlabel("Epoch")
    ax[0].grid(True, alpha=0.3)
    ax[0].legend()

    ax[1].plot(history["epoch"], history["val_map50"], color="green", linewidth=2, label="Val mAP@50 (EMA)")
    ax[1].plot(history["epoch"], history["precision"], linestyle="--", color="#ff7f0e", label="Precision@0.45")
    ax[1].plot(history["epoch"], history["recall"], linestyle="--", color="#2ca02c", label="Recall@0.45")
    ax[1].set_title(f"{name} - Validation Metrics (Best mAP@50: {best_ap:.4f})")
    ax[1].set_xlabel("Epoch")
    ax[1].set_ylim(0, 1.05)
    ax[1].grid(True, alpha=0.3)
    ax[1].legend()
    plt.tight_layout()
    plt.savefig(out_dir / "training_curves.png", dpi=150)
    plt.show()

    return best_ckpt

## Phần 2: Chạy Huấn luyện Module 1 (Xe) & Module 2 (Biển số trực tiếp) với D-FINE

In [ ]:
m1_ckpt = train_dfine_module(
    name="module1_vehicle",
    data_root=M1_DATA_ROOT,
    num_classes=len(VEHICLE_CLASSES),
    class_names=VEHICLE_CLASSES,
    epochs=50,
    warmup_epochs=5,
    batch_size=8 if DETECTOR_VARIANT == "dfine_l" else 16,
    lr=2e-4,
    img_size=(480, 800) if DETECTOR_VARIANT == "dfine_l" else (384, 640),
    num_queries=35,
    resume=True,
)

In [ ]:
m2_ckpt = train_dfine_module(
    name="module2_plate",
    data_root=M2_ROI_ROOT,
    num_classes=1,
    class_names=PLATE_CLASSES,
    epochs=50,
    warmup_epochs=5,
    batch_size=16,
    lr=2e-4,
    img_size=(256, 384),
    num_queries=15,
    resume=False,
)

   💾 [Train] Đã cache 1246 ảnh (576x1024) vào RAM (~2102.6 MB)
   💾 [Val] Đã cache 340 ảnh (576x1024) vào RAM (~573.8 MB)
🚀 Huấn luyện [module2_plate] | SOTA: DFINE_S | ImgSize: (576, 1024) | BS: 8 | Epochs: 80 (Warmup: 5) | Params: 7,258,132 | Train: 1246 | Val: 340
Epoch [01/80] (397.3s) | LR: 4.80e-05 | Loss: 11.8290 | Val mAP@50: 0.0039 | P@0.45: 0.0000 | R@0.45: 0.0000
Epoch [02/80] (905.5s) | LR: 8.60e-05 | Loss: 8.5579 | Val mAP@50: 0.0133 | P@0.45: 0.0000 | R@0.45: 0.0000
Epoch [03/80] (1222.6s) | LR: 1.24e-04 | Loss: 7.9398 | Val mAP@50: 0.0322 | P@0.45: 0.0000 | R@0.45: 0.0000
Epoch [04/80] (1309.6s) | LR: 1.62e-04 | Loss: 7.3162 | Val mAP@50: 0.1100 | P@0.45: 0.0000 | R@0.45: 0.0000
Epoch [05/80] (1305.8s) | LR: 2.00e-04 | Loss: 6.8977 | Val mAP@50: 0.1162 | P@0.45: 0.0000 | R@0.45: 0.0000
Epoch [06/80] (1306.5s) | LR: 2.00e-04 | Loss: 6.6392 | Val mAP@50: 0.1514 | P@0.45: 0.0000 | R@0.45: 0.0000
Epoch [07/80] (1299.5s) | LR: 2.00e-04 | Loss: 6.1443 | Val mAP@50: 0.2087 | P@

KeyboardInterrupt: 

## Phần 3: Dataset & Pipeline Huấn luyện Module 3 — SVTRv2 OCR (ICCV 2025, CTC-Based)

Pipeline huấn luyện **SVTRv2** đọc trực tiếp chuỗi ký tự biển số Việt Nam (hỗ trợ cả **biển 1 dòng** như ô tô `30A-123.45` và **biển 2 dòng** như xe máy `29-E1 / 123.45` thông qua hàm `unwrap_two_line_plate`):
* **Loss:** `nn.CTCLoss(blank=0, zero_infinity=True)` — không cần phân tách từng ký tự thủ công.
* **Metric:** `Plate Accuracy` (tỷ lệ nhận diện chính xác tuyệt đối 100% chuỗi biển số) và `Char Accuracy` ($1 - \text{CER}$).
* **Tối ưu:** RAM caching, ImageNet normalization, Linear Warmup + Cosine Annealing, và `ModelEMA`.

In [ ]:
# Khớp 100% với VN_PLATE_REGEX = re.compile(r"^[0-9]{2}[A-Z]{1,2}[0-9]{4,6}$") trong svtrv2_ocr.py
DEFAULT_PLATE_CHARSET = "0123456789ABCDEFGHKLMNPSTUVXYZ"


def ensure_svtr_sequence_labels(data_root: Path):
    """
    Nếu thư mục dữ liệu OCR đang dùng nhãn YOLO từng ký tự (kèm module3_ocr_data.yaml từ concat_data.py),
    tự động ghép các ký tự theo thứ tự trái->phải (hỗ trợ cả biển 2 dòng trên->dưới) thành labels.txt.
    """
    yaml_candidates = [data_root / "data.yaml", data_root.parent / "module3_ocr_data.yaml"]
    yaml_path = next((p for p in yaml_candidates if p.exists()), None)
    if yaml_path is None:
        return

    with open(yaml_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}
    names = cfg.get("names", [])
    if isinstance(names, dict):
        id_to_char = {int(k): str(v).strip().upper() for k, v in names.items()}
    else:
        id_to_char = {i: str(v).strip().upper() for i, v in enumerate(names)}

    for split in ["train", "val"]:
        split_dir = data_root / split
        txt_out = split_dir / "labels.txt"
        img_dir = split_dir / "images"
        lbl_dir = split_dir / "labels"
        if txt_out.exists() or not (img_dir.exists() and lbl_dir.exists()):
            continue

        lines_out = []
        for img_path in sorted(img_dir.iterdir()):
            if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
                continue
            lf = lbl_dir / f"{img_path.stem}.txt"
            if not lf.exists():
                continue
            chars = []
            for line in lf.read_text(encoding="utf-8").splitlines():
                parts = line.strip().split()
                if len(parts) >= 5:
                    cid = int(parts[0])
                    xc, yc, bw, bh = map(float, parts[1:5])
                    ch = id_to_char.get(cid, "")
                    if len(ch) == 1 and ch.isalnum():
                        chars.append((xc, yc, bh, ch))
            if not chars:
                continue

            # Phân tách biển 2 dòng nếu chênh lệch tọa độ Y lớn hơn 0.7 lần chiều cao ký tự trung bình
            ys = [c[1] for c in chars]
            avg_h = float(np.mean([c[2] for c in chars]))
            if (max(ys) - min(ys)) > max(0.18, 0.75 * avg_h):
                mid_y = 0.5 * (max(ys) + min(ys))
                top_row = sorted([c for c in chars if c[1] <= mid_y], key=lambda x: x[0])
                bot_row = sorted([c for c in chars if c[1] > mid_y], key=lambda x: x[0])
                ordered = top_row + bot_row
            else:
                ordered = sorted(chars, key=lambda x: x[0])

            plate_str = "".join([c[3] for c in ordered])
            if len(plate_str) >= 4:
                lines_out.append(f"{img_path.name}\t{plate_str}")

        if lines_out:
            txt_out.write_text("\n".join(lines_out), encoding="utf-8")
            print(f"[OCR {split}] Đã tự động ghép {len(lines_out)} chuỗi biển số vào {txt_out}")


def load_ocr_charset(data_root: Path) -> str:
    charset_file = data_root / "charset.txt"
    if charset_file.exists():
        chars = re.sub(r"[^A-Z0-9]", "", charset_file.read_text(encoding="utf-8").upper())
        if len(chars) > 0:
            return "".join(dict.fromkeys(chars))
    return DEFAULT_PLATE_CHARSET


def _safe_unwrap_plate(img_pil: Image.Image, target_hw=(48, 160)) -> Image.Image:
    img_bgr = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
    unwrapped = unwrap_two_line_plate(img_bgr, out_h=target_hw[0], out_w=target_hw[1])
    return Image.fromarray(cv2.cvtColor(unwrapped, cv2.COLOR_BGR2RGB))


class SVTROCRDataset(Dataset):
    def __init__(
        self,
        split_dir: Path,
        char2idx: dict,
        img_size=(48, 160),
        is_train: bool = True,
        cache_ram: bool = True,
        unwrap_2line: bool = True,
    ):
        self.split_dir = Path(split_dir)
        self.img_dir = self.split_dir / "images" if (self.split_dir / "images").exists() else self.split_dir
        self.char2idx = char2idx
        self.img_size = tuple(img_size)
        self.is_train = is_train
        self.cache_ram = cache_ram
        self.unwrap_2line = unwrap_2line

        if not self.img_dir.exists():
            raise FileNotFoundError(f"Không tìm thấy thư mục dữ liệu OCR: {self.img_dir}")

        self.samples = self._load_annotations()
        self.cached_imgs = []

        target_h, target_w = self.img_size
        total_bytes = 0
        if self.cache_ram:
            for img_path, _ in self.samples:
                with Image.open(img_path) as im:
                    img = im.convert("RGB")
                if self.unwrap_2line:
                    img = _safe_unwrap_plate(img, target_hw=self.img_size)
                img = TF.resize(img, [target_h, target_w])
                self.cached_imgs.append(img)
                total_bytes += target_h * target_w * 3

            ram_mb = total_bytes / (1024 ** 2)
            split_tag = "Train" if is_train else "Val"
            print(f"   [OCR {split_tag}] Đã cache {len(self.samples)} ảnh ({target_h}x{target_w}) vào RAM (~{ram_mb:.1f} MB)")

    def _clean_text(self, text: str) -> str:
        clean = re.sub(r"[^A-Z0-9]", "", text.strip().upper())
        return "".join([ch for ch in clean if ch in self.char2idx])

    def _load_annotations(self):
        img_map = {
            p.name: p
            for p in sorted(self.img_dir.iterdir())
            if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
        }
        stem_map = {p.stem: p for p in img_map.values()}
        samples = []

        txt_file = self.split_dir / "labels.txt"
        csv_file = self.split_dir / "labels.csv"
        lbl_folder = self.split_dir / "labels"

        if txt_file.exists():
            with open(txt_file, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    parts = line.split("\t") if "\t" in line else line.split(maxsplit=1)
                    if len(parts) == 2:
                        fname, raw_lbl = Path(parts[0]).name, parts[1]
                        p = img_map.get(fname) or stem_map.get(Path(fname).stem)
                        lbl = self._clean_text(raw_lbl)
                        if p is not None and len(lbl) > 0:
                            samples.append((p, lbl))
        elif csv_file.exists():
            df = pd.read_csv(csv_file)
            img_col = next((c for c in df.columns if c.lower() in ["filename", "image", "img", "file", "path"]), df.columns[0])
            lbl_col = next((c for c in df.columns if c.lower() in ["label", "text", "plate", "gt", "transcription"]), df.columns[1])
            for _, row in df.iterrows():
                fname = Path(str(row[img_col])).name
                p = img_map.get(fname) or stem_map.get(Path(fname).stem)
                lbl = self._clean_text(str(row[lbl_col]))
                if p is not None and len(lbl) > 0:
                    samples.append((p, lbl))
        elif lbl_folder.exists():
            for stem, p in stem_map.items():
                lf = lbl_folder / f"{stem}.txt"
                if lf.exists():
                    lbl = self._clean_text(lf.read_text(encoding="utf-8"))
                    if len(lbl) > 0:
                        samples.append((p, lbl))
        else:
            for stem, p in stem_map.items():
                raw_lbl = stem.split("_")[0]
                lbl = self._clean_text(raw_lbl)
                if len(lbl) >= 4:
                    samples.append((p, lbl))

        if len(samples) == 0:
            raise RuntimeError(f"Không đọc được mẫu nhãn OCR hợp lệ nào trong {self.split_dir}.")
        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, gt_text = self.samples[idx]
        if self.cache_ram:
            img = self.cached_imgs[idx].copy()
        else:
            with Image.open(img_path) as im:
                img = im.convert("RGB")
            if self.unwrap_2line:
                img = _safe_unwrap_plate(img, target_hw=self.img_size)
            img = TF.resize(img, list(self.img_size))

        if self.is_train:
            if random.random() < 0.5:
                img = TF.affine(
                    img,
                    angle=random.uniform(-4.0, 4.0),
                    translate=[0, 0],
                    scale=random.uniform(0.92, 1.04),
                    shear=[random.uniform(-4.0, 4.0), 0.0],
                    fill=[114, 114, 114],
                )
            if random.random() < 0.7:
                img = TF.adjust_brightness(img, random.uniform(0.60, 1.40))
                img = TF.adjust_contrast(img, random.uniform(0.65, 1.35))
                img = TF.adjust_saturation(img, random.uniform(0.70, 1.30))
            if random.random() < 0.2:
                img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.3, 1.1)))

        img_tensor = TF.to_tensor(img)
        img_tensor = TF.normalize(img_tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD)
        target_ids = torch.tensor([self.char2idx[c] for c in gt_text], dtype=torch.long)
        return img_tensor, target_ids, len(target_ids), gt_text


def svtr_collate_fn(batch):
    imgs, target_ids, target_lengths, gt_texts = zip(*batch)
    return (
        torch.stack(imgs, dim=0),
        torch.cat(target_ids, dim=0),
        torch.tensor(target_lengths, dtype=torch.long),
        list(gt_texts),
    )


def _levenshtein_distance(s1: str, s2: str) -> int:
    if len(s1) < len(s2):
        return _levenshtein_distance(s2, s1)
    if len(s2) == 0:
        return len(s1)
    prev_row = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1):
        curr_row = [i + 1]
        for j, c2 in enumerate(s2):
            curr_row.append(min(prev_row[j + 1] + 1, curr_row[j] + 1, prev_row[j] + (c1 != c2)))
        prev_row = curr_row
    return prev_row[-1]


@torch.no_grad()
def evaluate_svtr_ocr(model, val_loader, idx2char: dict, device):
    model.eval()
    exact_matches = 0
    total_plates = 0
    total_edit_dist = 0
    total_chars = 0
    sample_pairs = []

    for imgs, _, _, gt_texts in val_loader:
        log_probs = model(imgs.to(device))  # [T, B, num_classes]
        decoded = ctc_greedy_decode(log_probs, idx2char)  # [(text, conf), ...]

        for gt, (pred, _) in zip(gt_texts, decoded):
            total_plates += 1
            if pred == gt:
                exact_matches += 1
            total_edit_dist += _levenshtein_distance(gt, pred)
            total_chars += max(1, len(gt))
            if len(sample_pairs) < 4:
                sample_pairs.append((gt, pred))

    plate_acc = exact_matches / max(1, total_plates)
    cer = total_edit_dist / max(1, total_chars)
    char_acc = max(0.0, 1.0 - cer)
    return float(plate_acc), float(char_acc), sample_pairs


def train_svtrv2_ocr(
    name: str,
    data_root: Path,
    epochs: int = 60,
    warmup_epochs: int = 5,
    batch_size: int = 32,
    lr: float = 5e-4,
    img_size=(48, 160),
    ema_decay: float = 0.9995,
    resume: bool = False,
):
    ensure_svtr_sequence_labels(data_root)
    charset = load_ocr_charset(data_root)
    char2idx = {ch: i + 1 for i, ch in enumerate(charset)}
    idx2char = {i + 1: ch for i, ch in enumerate(charset)}
    num_classes = len(charset) + 1

    train_ds = SVTROCRDataset(data_root / "train", char2idx=char2idx, img_size=img_size, is_train=True)
    val_ds = SVTROCRDataset(data_root / "val", char2idx=char2idx, img_size=img_size, is_train=False)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY, collate_fn=svtr_collate_fn
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY, collate_fn=svtr_collate_fn
    )

    sig = inspect.signature(build_svtrv2_ocr)
    build_kwargs = {"num_classes": num_classes}
    if "variant" in sig.parameters:
        build_kwargs["variant"] = OCR_VARIANT
    model = build_svtrv2_ocr(**build_kwargs).to(DEVICE)

    ctc_loss_fn = nn.CTCLoss(blank=0, zero_infinity=True)
    ema = ModelEMA(model, decay=ema_decay)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    warmup_epochs = min(warmup_epochs, max(1, epochs // 4))
    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.05, end_factor=1.0, total_iters=warmup_epochs
    )
    cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(1, epochs - warmup_epochs), eta_min=lr * 0.01
    )
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs]
    )
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

    out_dir = SAVE_DIR / name
    out_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt = out_dir / f"{name}_{OCR_VARIANT}_best.pth"
    last_ckpt = out_dir / f"{name}_{OCR_VARIANT}_last.pth"

    start_epoch = 1
    best_plate_acc = -1.0
    if resume and last_ckpt.exists():
        ckpt = torch.load(last_ckpt, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["raw_model_state"])
        ema.ema_model.load_state_dict(ckpt["model_state"])
        if "optimizer_state" in ckpt:
            optimizer.load_state_dict(ckpt["optimizer_state"])
        if "scheduler_state" in ckpt:
            scheduler.load_state_dict(ckpt["scheduler_state"])
        start_epoch = int(ckpt.get("epoch", 0)) + 1
        best_plate_acc = float(ckpt.get("best_plate_acc", -1.0))

    history = {"epoch": [], "lr": [], "train_ctc_loss": [], "val_plate_acc": [], "val_char_acc": []}
    total_params = sum(p.numel() for p in model.parameters())
    print(
        f"Huấn luyện [{name}] | SOTA: {OCR_VARIANT.upper()} | ImgSize: {img_size} | "
        f"Charset ({len(charset)}): '{charset}' | Params: {total_params:,} | Train: {len(train_ds)} | Val: {len(val_ds)}"
    )

    for epoch in range(start_epoch, epochs + 1):
        model.train()
        total_loss = 0.0
        t0 = time.perf_counter()

        for step, (imgs, targets_concat, target_lengths, _) in enumerate(train_loader, 1):
            imgs = imgs.to(DEVICE, non_blocking=True)
            targets_concat = targets_concat.to(DEVICE, non_blocking=True)
            target_lengths = target_lengths.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                # SVTRv2OCR.forward() trả về trực tiếp log_probs shape [T, B, num_classes]
                log_probs = model(imgs).float()
                T, B, _ = log_probs.shape
                input_lengths = torch.full((B,), T, dtype=torch.long, device=DEVICE)
                loss = ctc_loss_fn(log_probs, targets_concat, input_lengths, target_lengths)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()

            ema.update(model)
            total_loss += loss.item()

        scheduler.step()
        current_lr = optimizer.param_groups[0]["lr"]
        avg_loss = total_loss / max(1, len(train_loader))

        plate_acc, char_acc, samples = evaluate_svtr_ocr(ema.ema_model, val_loader, idx2char, DEVICE)
        dt = time.perf_counter() - t0

        history["epoch"].append(epoch)
        history["lr"].append(current_lr)
        history["train_ctc_loss"].append(avg_loss)
        history["val_plate_acc"].append(plate_acc)
        history["val_char_acc"].append(char_acc)

        sample_str = " | ".join([f"{gt}->{pr}" for gt, pr in samples[:3]])
        print(
            f"Epoch [{epoch:02d}/{epochs:02d}] ({dt:.1f}s) | Loss: {avg_loss:.4f} | "
            f"Plate Acc: {plate_acc * 100:.2f}% | Char Acc: {char_acc * 100:.2f}% | Mẫu: [{sample_str}]"
        )

        ckpt_payload = {
            "variant": OCR_VARIANT,
            "charset": charset,
            "char2idx": char2idx,
            "idx2char": idx2char,
            "num_classes": num_classes,
            "img_size": img_size,
            "mean": IMAGENET_MEAN,
            "std": IMAGENET_STD,
            "epoch": epoch,
            "best_plate_acc": max(best_plate_acc, plate_acc),
            "model_state": ema.ema_model.state_dict(),
            "raw_model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
        }
        torch.save(ckpt_payload, last_ckpt)

        if plate_acc > best_plate_acc:
            best_plate_acc = plate_acc
            torch.save(ckpt_payload, best_ckpt)

    pd.DataFrame(history).to_csv(out_dir / "history.csv", index=False)

    fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
    ax[0].plot(history["epoch"], history["train_ctc_loss"], label="SVTRv2 CTC Loss", color="#1f77b4")
    ax[0].set_title(f"{name} ({OCR_VARIANT}) - Training CTC Loss")
    ax[0].set_xlabel("Epoch")
    ax[0].grid(True, alpha=0.3)
    ax[0].legend()

    ax[1].plot(history["epoch"], [x * 100 for x in history["val_plate_acc"]], color="green", linewidth=2, label="Plate Accuracy (%)")
    ax[1].plot(history["epoch"], [x * 100 for x in history["val_char_acc"]], linestyle="--", color="#ff7f0e", label="Char Accuracy (%)")
    ax[1].set_title(f"{name} - Validation OCR Accuracy (Best Plate Acc: {best_plate_acc * 100:.2f}%)")
    ax[1].set_xlabel("Epoch")
    ax[1].set_ylim(0, 105)
    ax[1].grid(True, alpha=0.3)
    ax[1].legend()
    plt.tight_layout()
    plt.savefig(out_dir / "training_curves.png", dpi=150)
    plt.show()

    return best_ckpt


ocr_data_dir = next(
    (PROCESSED_DIR / d for d in ["module3_plate_ocr", "module3_ocr", "module3_ocr_det", "plate_ocr", "ocr"] if (PROCESSED_DIR / d).exists()),
    PROCESSED_DIR / "module3_plate_ocr",
)

if ocr_data_dir.exists():
    m3_ckpt = train_svtrv2_ocr(
        name="module3_ocr",
        data_root=ocr_data_dir,
        epochs=60,
        warmup_epochs=5,
        batch_size=32,
        lr=5e-4,
        img_size=(48, 160),
        resume=False,
    )
else:
    m3_ckpt = None
    print(f"Chưa tìm thấy thư mục dữ liệu OCR tại {ocr_data_dir}.")

In [ ]:
# Khớp 100% với VN_PLATE_REGEX = re.compile(r"^[0-9]{2}[A-Z]{1,2}[0-9]{4,6}$") trong svtrv2_ocr.py
DEFAULT_PLATE_CHARSET = "0123456789ABCDEFGHKLMNPSTUVXYZ"


def ensure_svtr_sequence_labels(data_root: Path):
    """
    Nếu thư mục dữ liệu OCR đang dùng nhãn YOLO từng ký tự (kèm module3_ocr_data.yaml từ concat_data.py),
    tự động ghép các ký tự theo thứ tự trái->phải (hỗ trợ cả biển 2 dòng trên->dưới) thành labels.txt.
    """
    yaml_candidates = [data_root / "data.yaml", data_root.parent / "module3_ocr_data.yaml"]
    yaml_path = next((p for p in yaml_candidates if p.exists()), None)
    if yaml_path is None:
        return

    with open(yaml_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}
    names = cfg.get("names", [])
    if isinstance(names, dict):
        id_to_char = {int(k): str(v).strip().upper() for k, v in names.items()}
    else:
        id_to_char = {i: str(v).strip().upper() for i, v in enumerate(names)}

    for split in ["train", "val"]:
        split_dir = data_root / split
        txt_out = split_dir / "labels.txt"
        img_dir = split_dir / "images"
        lbl_dir = split_dir / "labels"
        if txt_out.exists() or not (img_dir.exists() and lbl_dir.exists()):
            continue

        lines_out = []
        for img_path in sorted(img_dir.iterdir()):
            if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
                continue
            lf = lbl_dir / f"{img_path.stem}.txt"
            if not lf.exists():
                continue
            chars = []
            for line in lf.read_text(encoding="utf-8").splitlines():
                parts = line.strip().split()
                if len(parts) >= 5:
                    cid = int(parts[0])
                    xc, yc, bw, bh = map(float, parts[1:5])
                    ch = id_to_char.get(cid, "")
                    if len(ch) == 1 and ch.isalnum():
                        chars.append((xc, yc, bh, ch))
            if not chars:
                continue

            # Phân tách biển 2 dòng nếu chênh lệch tọa độ Y lớn hơn 0.7 lần chiều cao ký tự trung bình
            ys = [c[1] for c in chars]
            avg_h = float(np.mean([c[2] for c in chars]))
            if (max(ys) - min(ys)) > max(0.18, 0.75 * avg_h):
                mid_y = 0.5 * (max(ys) + min(ys))
                top_row = sorted([c for c in chars if c[1] <= mid_y], key=lambda x: x[0])
                bot_row = sorted([c for c in chars if c[1] > mid_y], key=lambda x: x[0])
                ordered = top_row + bot_row
            else:
                ordered = sorted(chars, key=lambda x: x[0])

            plate_str = "".join([c[3] for c in ordered])
            if len(plate_str) >= 4:
                lines_out.append(f"{img_path.name}\t{plate_str}")

        if lines_out:
            txt_out.write_text("\n".join(lines_out), encoding="utf-8")
            print(f"[OCR {split}] Đã tự động ghép {len(lines_out)} chuỗi biển số vào {txt_out}")


def load_ocr_charset(data_root: Path) -> str:
    charset_file = data_root / "charset.txt"
    if charset_file.exists():
        chars = re.sub(r"[^A-Z0-9]", "", charset_file.read_text(encoding="utf-8").upper())
        if len(chars) > 0:
            return "".join(dict.fromkeys(chars))
    return DEFAULT_PLATE_CHARSET


def _safe_unwrap_plate(img_pil: Image.Image, target_hw=(48, 160)) -> Image.Image:
    img_bgr = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
    unwrapped = unwrap_two_line_plate(img_bgr, out_h=target_hw[0], out_w=target_hw[1])
    return Image.fromarray(cv2.cvtColor(unwrapped, cv2.COLOR_BGR2RGB))


class SVTROCRDataset(Dataset):
    def __init__(
        self,
        split_dir: Path,
        char2idx: dict,
        img_size=(48, 160),
        is_train: bool = True,
        cache_ram: bool = True,
        unwrap_2line: bool = True,
    ):
        self.split_dir = Path(split_dir)
        self.img_dir = self.split_dir / "images" if (self.split_dir / "images").exists() else self.split_dir
        self.char2idx = char2idx
        self.img_size = tuple(img_size)
        self.is_train = is_train
        self.cache_ram = cache_ram
        self.unwrap_2line = unwrap_2line

        if not self.img_dir.exists():
            raise FileNotFoundError(f"Không tìm thấy thư mục dữ liệu OCR: {self.img_dir}")

        self.samples = self._load_annotations()
        self.cached_imgs = []

        target_h, target_w = self.img_size
        total_bytes = 0
        if self.cache_ram:
            for img_path, _ in self.samples:
                with Image.open(img_path) as im:
                    img = im.convert("RGB")
                if self.unwrap_2line:
                    img = _safe_unwrap_plate(img, target_hw=self.img_size)
                img = TF.resize(img, [target_h, target_w])
                self.cached_imgs.append(img)
                total_bytes += target_h * target_w * 3

            ram_mb = total_bytes / (1024 ** 2)
            split_tag = "Train" if is_train else "Val"
            print(f"   [OCR {split_tag}] Đã cache {len(self.samples)} ảnh ({target_h}x{target_w}) vào RAM (~{ram_mb:.1f} MB)")

    def _clean_text(self, text: str) -> str:
        clean = re.sub(r"[^A-Z0-9]", "", text.strip().upper())
        return "".join([ch for ch in clean if ch in self.char2idx])

    def _load_annotations(self):
        img_map = {
            p.name: p
            for p in sorted(self.img_dir.iterdir())
            if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
        }
        stem_map = {p.stem: p for p in img_map.values()}
        samples = []

        txt_file = self.split_dir / "labels.txt"
        csv_file = self.split_dir / "labels.csv"
        lbl_folder = self.split_dir / "labels"

        if txt_file.exists():
            with open(txt_file, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    parts = line.split("\t") if "\t" in line else line.split(maxsplit=1)
                    if len(parts) == 2:
                        fname, raw_lbl = Path(parts[0]).name, parts[1]
                        p = img_map.get(fname) or stem_map.get(Path(fname).stem)
                        lbl = self._clean_text(raw_lbl)
                        if p is not None and len(lbl) > 0:
                            samples.append((p, lbl))
        elif csv_file.exists():
            df = pd.read_csv(csv_file)
            img_col = next((c for c in df.columns if c.lower() in ["filename", "image", "img", "file", "path"]), df.columns[0])
            lbl_col = next((c for c in df.columns if c.lower() in ["label", "text", "plate", "gt", "transcription"]), df.columns[1])
            for _, row in df.iterrows():
                fname = Path(str(row[img_col])).name
                p = img_map.get(fname) or stem_map.get(Path(fname).stem)
                lbl = self._clean_text(str(row[lbl_col]))
                if p is not None and len(lbl) > 0:
                    samples.append((p, lbl))
        elif lbl_folder.exists():
            for stem, p in stem_map.items():
                lf = lbl_folder / f"{stem}.txt"
                if lf.exists():
                    lbl = self._clean_text(lf.read_text(encoding="utf-8"))
                    if len(lbl) > 0:
                        samples.append((p, lbl))
        else:
            for stem, p in stem_map.items():
                raw_lbl = stem.split("_")[0]
                lbl = self._clean_text(raw_lbl)
                if len(lbl) >= 4:
                    samples.append((p, lbl))

        if len(samples) == 0:
            raise RuntimeError(f"Không đọc được mẫu nhãn OCR hợp lệ nào trong {self.split_dir}.")
        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, gt_text = self.samples[idx]
        if self.cache_ram:
            img = self.cached_imgs[idx].copy()
        else:
            with Image.open(img_path) as im:
                img = im.convert("RGB")
            if self.unwrap_2line:
                img = _safe_unwrap_plate(img, target_hw=self.img_size)
            img = TF.resize(img, list(self.img_size))

        if self.is_train:
            if random.random() < 0.5:
                img = TF.affine(
                    img,
                    angle=random.uniform(-4.0, 4.0),
                    translate=[0, 0],
                    scale=random.uniform(0.92, 1.04),
                    shear=[random.uniform(-4.0, 4.0), 0.0],
                    fill=[114, 114, 114],
                )
            if random.random() < 0.7:
                img = TF.adjust_brightness(img, random.uniform(0.60, 1.40))
                img = TF.adjust_contrast(img, random.uniform(0.65, 1.35))
                img = TF.adjust_saturation(img, random.uniform(0.70, 1.30))
            if random.random() < 0.2:
                img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.3, 1.1)))

        img_tensor = TF.to_tensor(img)
        img_tensor = TF.normalize(img_tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD)
        target_ids = torch.tensor([self.char2idx[c] for c in gt_text], dtype=torch.long)
        return img_tensor, target_ids, len(target_ids), gt_text


def svtr_collate_fn(batch):
    imgs, target_ids, target_lengths, gt_texts = zip(*batch)
    return (
        torch.stack(imgs, dim=0),
        torch.cat(target_ids, dim=0),
        torch.tensor(target_lengths, dtype=torch.long),
        list(gt_texts),
    )


def _levenshtein_distance(s1: str, s2: str) -> int:
    if len(s1) < len(s2):
        return _levenshtein_distance(s2, s1)
    if len(s2) == 0:
        return len(s1)
    prev_row = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1):
        curr_row = [i + 1]
        for j, c2 in enumerate(s2):
            curr_row.append(min(prev_row[j + 1] + 1, curr_row[j] + 1, prev_row[j] + (c1 != c2)))
        prev_row = curr_row
    return prev_row[-1]


@torch.no_grad()
def evaluate_svtr_ocr(model, val_loader, idx2char: dict, device):
    model.eval()
    exact_matches = 0
    total_plates = 0
    total_edit_dist = 0
    total_chars = 0
    sample_pairs = []

    for imgs, _, _, gt_texts in val_loader:
        log_probs = model(imgs.to(device))  # [T, B, num_classes]
        decoded = ctc_greedy_decode(log_probs, idx2char)  # [(text, conf), ...]

        for gt, (pred, _) in zip(gt_texts, decoded):
            total_plates += 1
            if pred == gt:
                exact_matches += 1
            total_edit_dist += _levenshtein_distance(gt, pred)
            total_chars += max(1, len(gt))
            if len(sample_pairs) < 4:
                sample_pairs.append((gt, pred))

    plate_acc = exact_matches / max(1, total_plates)
    cer = total_edit_dist / max(1, total_chars)
    char_acc = max(0.0, 1.0 - cer)
    return float(plate_acc), float(char_acc), sample_pairs


def train_svtrv2_ocr(
    name: str,
    data_root: Path,
    epochs: int = 60,
    warmup_epochs: int = 5,
    batch_size: int = 32,
    lr: float = 5e-4,
    img_size=(48, 160),
    ema_decay: float = 0.9995,
    resume: bool = False,
):
    ensure_svtr_sequence_labels(data_root)
    charset = load_ocr_charset(data_root)
    char2idx = {ch: i + 1 for i, ch in enumerate(charset)}
    idx2char = {i + 1: ch for i, ch in enumerate(charset)}
    num_classes = len(charset) + 1

    train_ds = SVTROCRDataset(data_root / "train", char2idx=char2idx, img_size=img_size, is_train=True)
    val_ds = SVTROCRDataset(data_root / "val", char2idx=char2idx, img_size=img_size, is_train=False)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY, collate_fn=svtr_collate_fn
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY, collate_fn=svtr_collate_fn
    )

    sig = inspect.signature(build_svtrv2_ocr)
    build_kwargs = {"num_classes": num_classes}
    if "variant" in sig.parameters:
        build_kwargs["variant"] = OCR_VARIANT
    model = build_svtrv2_ocr(**build_kwargs).to(DEVICE)

    ctc_loss_fn = nn.CTCLoss(blank=0, zero_infinity=True)
    ema = ModelEMA(model, decay=ema_decay)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    warmup_epochs = min(warmup_epochs, max(1, epochs // 4))
    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.05, end_factor=1.0, total_iters=warmup_epochs
    )
    cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(1, epochs - warmup_epochs), eta_min=lr * 0.01
    )
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs]
    )
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

    out_dir = SAVE_DIR / name
    out_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt = out_dir / f"{name}_{OCR_VARIANT}_best.pth"
    last_ckpt = out_dir / f"{name}_{OCR_VARIANT}_last.pth"

    start_epoch = 1
    best_plate_acc = -1.0
    if resume and last_ckpt.exists():
        ckpt = torch.load(last_ckpt, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["raw_model_state"])
        ema.ema_model.load_state_dict(ckpt["model_state"])
        if "optimizer_state" in ckpt:
            optimizer.load_state_dict(ckpt["optimizer_state"])
        if "scheduler_state" in ckpt:
            scheduler.load_state_dict(ckpt["scheduler_state"])
        start_epoch = int(ckpt.get("epoch", 0)) + 1
        best_plate_acc = float(ckpt.get("best_plate_acc", -1.0))

    history = {"epoch": [], "lr": [], "train_ctc_loss": [], "val_plate_acc": [], "val_char_acc": []}
    total_params = sum(p.numel() for p in model.parameters())
    print(
        f"Huấn luyện [{name}] | SOTA: {OCR_VARIANT.upper()} | ImgSize: {img_size} | "
        f"Charset ({len(charset)}): '{charset}' | Params: {total_params:,} | Train: {len(train_ds)} | Val: {len(val_ds)}"
    )

    for epoch in range(start_epoch, epochs + 1):
        model.train()
        total_loss = 0.0
        t0 = time.perf_counter()

        for step, (imgs, targets_concat, target_lengths, _) in enumerate(train_loader, 1):
            imgs = imgs.to(DEVICE, non_blocking=True)
            targets_concat = targets_concat.to(DEVICE, non_blocking=True)
            target_lengths = target_lengths.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                # SVTRv2OCR.forward() trả về trực tiếp log_probs shape [T, B, num_classes]
                log_probs = model(imgs).float()
                T, B, _ = log_probs.shape
                input_lengths = torch.full((B,), T, dtype=torch.long, device=DEVICE)
                loss = ctc_loss_fn(log_probs, targets_concat, input_lengths, target_lengths)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()

            ema.update(model)
            total_loss += loss.item()

        scheduler.step()
        current_lr = optimizer.param_groups[0]["lr"]
        avg_loss = total_loss / max(1, len(train_loader))

        plate_acc, char_acc, samples = evaluate_svtr_ocr(ema.ema_model, val_loader, idx2char, DEVICE)
        dt = time.perf_counter() - t0

        history["epoch"].append(epoch)
        history["lr"].append(current_lr)
        history["train_ctc_loss"].append(avg_loss)
        history["val_plate_acc"].append(plate_acc)
        history["val_char_acc"].append(char_acc)

        sample_str = " | ".join([f"{gt}->{pr}" for gt, pr in samples[:3]])
        print(
            f"Epoch [{epoch:02d}/{epochs:02d}] ({dt:.1f}s) | Loss: {avg_loss:.4f} | "
            f"Plate Acc: {plate_acc * 100:.2f}% | Char Acc: {char_acc * 100:.2f}% | Mẫu: [{sample_str}]"
        )

        ckpt_payload = {
            "variant": OCR_VARIANT,
            "charset": charset,
            "char2idx": char2idx,
            "idx2char": idx2char,
            "num_classes": num_classes,
            "img_size": img_size,
            "mean": IMAGENET_MEAN,
            "std": IMAGENET_STD,
            "epoch": epoch,
            "best_plate_acc": max(best_plate_acc, plate_acc),
            "model_state": ema.ema_model.state_dict(),
            "raw_model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
        }
        torch.save(ckpt_payload, last_ckpt)

        if plate_acc > best_plate_acc:
            best_plate_acc = plate_acc
            torch.save(ckpt_payload, best_ckpt)

    pd.DataFrame(history).to_csv(out_dir / "history.csv", index=False)

    fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
    ax[0].plot(history["epoch"], history["train_ctc_loss"], label="SVTRv2 CTC Loss", color="#1f77b4")
    ax[0].set_title(f"{name} ({OCR_VARIANT}) - Training CTC Loss")
    ax[0].set_xlabel("Epoch")
    ax[0].grid(True, alpha=0.3)
    ax[0].legend()

    ax[1].plot(history["epoch"], [x * 100 for x in history["val_plate_acc"]], color="green", linewidth=2, label="Plate Accuracy (%)")
    ax[1].plot(history["epoch"], [x * 100 for x in history["val_char_acc"]], linestyle="--", color="#ff7f0e", label="Char Accuracy (%)")
    ax[1].set_title(f"{name} - Validation OCR Accuracy (Best Plate Acc: {best_plate_acc * 100:.2f}%)")
    ax[1].set_xlabel("Epoch")
    ax[1].set_ylim(0, 105)
    ax[1].grid(True, alpha=0.3)
    ax[1].legend()
    plt.tight_layout()
    plt.savefig(out_dir / "training_curves.png", dpi=150)
    plt.show()

    return best_ckpt


ocr_data_dir = next(
    (PROCESSED_DIR / d for d in ["module3_plate_ocr", "module3_ocr", "module3_ocr_det", "plate_ocr", "ocr"] if (PROCESSED_DIR / d).exists()),
    PROCESSED_DIR / "module3_plate_ocr",
)

if ocr_data_dir.exists():
    m3_ckpt = train_svtrv2_ocr(
        name="module3_ocr",
        data_root=ocr_data_dir,
        epochs=60,
        warmup_epochs=5,
        batch_size=32,
        lr=5e-4,
        img_size=(48, 160),
        resume=False,
    )
else:
    m3_ckpt = None
    print(f"Chưa tìm thấy thư mục dữ liệu OCR tại {ocr_data_dir}.")

## Phần 4: Kiểm tra Chuẩn hóa Inference (Sanity Check) & Tổng hợp Kết quả Huấn luyện

In [ ]:
def preprocess_bgr_tensor(img_bgr: np.ndarray, img_size: tuple) -> torch.Tensor:
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(rgb)
    resized = TF.resize(pil_img, list(img_size))
    tensor = TF.to_tensor(resized)
    return TF.normalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD).unsqueeze(0)


@torch.no_grad()
def run_edge_alpr_sanity_check(frame_bgr: np.ndarray, m1_path: Path, m2_path: Path, m3_path: Path):
    ckpt1 = torch.load(m1_path, map_location=DEVICE, weights_only=False)
    m1 = build_dfine_detector(
        num_classes=ckpt1["num_classes"],
        variant=ckpt1.get("variant", DETECTOR_VARIANT),
        pretrained_backbone=False,
        num_queries=ckpt1.get("num_queries", 35),
    ).to(DEVICE).eval()
    m1.load_state_dict(ckpt1["model_state"])

    ckpt2 = torch.load(m2_path, map_location=DEVICE, weights_only=False)
    m2 = build_dfine_detector(
        num_classes=ckpt2["num_classes"],
        variant=ckpt2.get("variant", DETECTOR_VARIANT),
        pretrained_backbone=False,
        num_queries=ckpt2.get("num_queries", 15),
    ).to(DEVICE).eval()
    m2.load_state_dict(ckpt2["model_state"])

    ckpt3 = torch.load(m3_path, map_location=DEVICE, weights_only=False)
    m3 = build_svtrv2_ocr(
        num_classes=ckpt3["num_classes"],
        variant=ckpt3.get("variant", OCR_VARIANT),
    ).to(DEVICE).eval()
    m3.load_state_dict(ckpt3["model_state"])
    idx2char = ckpt3["idx2char"]

    tracker = DashcamHybridTracker(single_class_mode=(ckpt1["num_classes"] == 1))
    voter = PlateVoteAggregator()

    H, W = frame_bgr.shape[:2]
    in1_h, in1_w = ckpt1["img_size"]
    det1 = m1.predict(preprocess_bgr_tensor(frame_bgr, (in1_h, in1_w)).to(DEVICE), score_thresh=0.40)[0]

    boxes1 = det1["boxes"].cpu().numpy()
    if len(boxes1) > 0:
        boxes1[:, [0, 2]] *= (W / in1_w)
        boxes1[:, [1, 3]] *= (H / in1_h)
    scores1 = det1["scores"].cpu().numpy()
    labels1 = det1["labels"].cpu().numpy()
    if len(labels1) > 0 and labels1.max() >= ckpt1["num_classes"]:
        labels1 = labels1 - 1

    tracks = tracker.update(boxes1, scores1, labels1, frame_bgr=frame_bgr, now_ts=time.time())
    priority_tracks = select_priority_tracks(tracks, W, H, max_k=2, locked_ids=voter.locked_ids)

    results = []
    in2_h, in2_w = ckpt2["img_size"]
    in3_h, in3_w = ckpt3["img_size"]

    for trk in priority_tracks:
        tid, vbox = trk[0], trk[1]
        roi_bgr, (rx1, ry1, rx2, ry2) = extract_plate_search_roi(frame_bgr, vbox)
        if roi_bgr.size == 0:
            continue

        det2 = m2.predict(preprocess_bgr_tensor(roi_bgr, (in2_h, in2_w)).to(DEVICE), score_thresh=0.45)[0]
        if len(det2["boxes"]) == 0:
            continue

        best_idx = int(torch.argmax(det2["scores"]).item())
        pbox = det2["boxes"][best_idx].cpu().numpy()
        rh, rw = roi_bgr.shape[:2]
        px1 = max(0, int(round(pbox[0] * (rw / in2_w))))
        py1 = max(0, int(round(pbox[1] * (rh / in2_h))))
        px2 = min(rw, int(round(pbox[2] * (rw / in2_w))))
        py2 = min(rh, int(round(pbox[3] * (rh / in2_h))))
        plate_crop = roi_bgr[py1:py2, px1:px2]
        if plate_crop.size == 0:
            continue

        sharpness = compute_plate_sharpness(plate_crop)
        unwrapped = unwrap_two_line_plate(plate_crop, out_h=in3_h, out_w=in3_w)
        log_probs = m3(preprocess_bgr_tensor(unwrapped, (in3_h, in3_w)).to(DEVICE))
        pred_text, ocr_conf = ctc_greedy_decode(log_probs, idx2char)[0]
        voted = voter.update(tid, pred_text, ocr_conf, sharpness)
        results.append({"track_id": tid, "raw_ocr": pred_text, "conf": round(ocr_conf, 3), "voted": voted})

    return results


# Tổng hợp bảng kết quả huấn luyện tốt nhất của cả 3 module
summary_rows = []
for mod_name, metric_col, metric_label in [
    ("module1_vehicle", "val_map50", "Best Val mAP@50"),
    ("module2_plate", "val_map50", "Best Val mAP@50"),
    ("module3_ocr", "val_plate_acc", "Best Val Plate Accuracy"),
]:
    hist_path = SAVE_DIR / mod_name / "history.csv"
    if hist_path.exists():
        df_h = pd.read_csv(hist_path)
        best_idx = df_h[metric_col].idxmax()
        best_row = df_h.loc[best_idx]
        summary_rows.append({
            "Module": mod_name,
            "Best Epoch": int(best_row["epoch"]),
            "Metric": metric_label,
            "Best Value": round(float(best_row[metric_col]), 4),
            "P@0.45": round(float(best_row["precision"]), 4) if "precision" in best_row else "-",
            "R@0.45": round(float(best_row["recall"]), 4) if "recall" in best_row else "-",
        })

if summary_rows:
    display(pd.DataFrame(summary_rows))
else:
    print("Chạy xong các cell huấn luyện ở trên để xem bảng tổng hợp chỉ số tại đây.")